[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/100.%20The%20Closed-Loop%20%28Reverse%20Logistics%29%20Network%20Design%20Problem/P100-Tier-7_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 100. The Closed-Loop Network Design Problem
## Tier 7: The Human–AI Symbiotic Partnership

### Purpose and scope
This notebook presents a **simulation of a human–AI decision partnership** in the context of logistics network design. A human expert and an explainable-AI stand-in make decisions, their recommendations are combined using confidence- and trust-based weights, and feedback updates selected partnership parameters. The notebook demonstrates the partnership framework; it does **not** formulate or solve a closed-loop facility-location, flow, or reverse-logistics optimization problem.

The inputs are synthetic decision contexts and configurable expert, AI, and trust-framework parameters. The outputs are simulated decisions, trust and performance indicators, a partnership report, and visualizations. A fixed random seed makes the stochastic draws reproducible when the cells are run in order. Timestamps are informational and are not reproducible.

### Modeling assumptions
- Human and AI recommendations are represented by three comparable scores: risk tolerance, innovation preference, and complexity handling. These are illustrative decision attributes, not facility locations, routes, quantities, or other operational decisions.
- The AI component is a transparent **simulation**, not a trained neural network or a production optimization model. Its pattern scores and recommendation attributes are generated synthetically.
- Decision quality, confidence, trust, and outcomes are normalized to the interval $[0,1]$. They are model indicators rather than empirically validated measures.
- Simulated outcomes provide feedback to the human expert's confidence and experience and to the AI component's learning-rate and explanation-level settings. The learning-rate update is a proxy; no model parameters are trained.
- The trust score is a weighted average of competence, confidence-based integrity proxies, and shared-goal alignment. This compact model omits many real-world factors, including accountability, privacy, organizational incentives, data governance, and operational constraints.

### Concepts and mathematical model
A decision is a choice made in a specified context. Here, each participant returns a vector of three scores, rather than an operational logistics plan. **Decision fusion** combines corresponding scores using normalized participant weights. **Trust** determines the initial balance between human and AI influence; confidence then adjusts that balance.

For human competence $q_H$, AI competence $q_A$, human confidence $c_H$, AI confidence $c_A$, shared values $v$, and goal alignment $g$, the trust score is

$$
T = \frac{0.3q_H + 0.3q_A + 0.2c_H + 0.2c_A + 0.2\left(\frac{v+g}{2}\right)}{1.2}.
$$

The denominator normalizes the stated component weights, whose sum is $1.2$, so that $T$ remains a weighted average on the $[0,1]$ scale. The code implements this calculation in `TrustFramework.calculate_trust_score`. Competence is represented by the most recent decision-quality score; integrity is approximated by confidence. These are modeling proxies, not direct measurements of competence or integrity.

The trust-dependent base weights $(b_H,b_A)$ are $(0.3,0.7)$ when $T>0.8$, $(0.5,0.5)$ when $0.5<T\leq0.8$, and $(0.7,0.3)$ otherwise. If participant confidences are $c_H$ and $c_A$, the normalized fusion weights are

$$
w_H = \frac{b_Hc_H}{b_Hc_H+b_Ac_A}, \qquad
w_A = \frac{b_Ac_A}{b_Hc_H+b_Ac_A}.
$$

For each shared decision attribute $k$, the fused score is

$$
x_k = w_H x_{H,k} + w_A x_{A,k}, \qquad w_H+w_A=1.
$$

The code implements these equations in `HumanAIPartnership._fuse_decisions`. If both confidence-adjusted weights are zero, it uses equal weights so the calculation remains defined. The fused decision-quality indicator is the same weighted average of the participants' quality indicators.

Feedback closes the simulation loop: a successful outcome raises the human expert's confidence and experience and increases the AI learning-rate proxy; an unsuccessful outcome decreases confidence and the AI learning-rate proxy. Thus, the simulation follows the sequence **individual recommendations → trust assessment → weighted fusion → synthetic outcome → feedback**. It does not establish that collaboration outperforms either participant acting alone; such a claim would require a designed comparative experiment and evidence.

### Approach
1. Define the human expert, explainable-AI simulation, trust framework, and partnership.
2. Generate synthetic contexts and participant recommendations.
3. Calculate trust and confidence-adjusted fusion weights, then combine matching decision attributes.
4. Simulate an outcome and use it as feedback to update partnership settings.
5. Summarize the simulated decisions, metrics, partnership assessment, and trends.

### Interpreting results
Scores describe behavior under the assumptions and random draws of this simulation. A high trust score indicates a high value under the specified weighted proxy; it is not proof of calibrated trust or decision correctness. A high simulated success rate is not evidence of real-world effectiveness. Read the plots alongside the model definitions, and do not interpret the illustrative recommendation scores as a feasible network design.

In [ ]:
from dataclasses import dataclass, field
from enum import Enum
import time
from typing import Any, Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np

# Fix both random generators so synthetic draws can be reproduced in a fresh run.
np.random.seed(42)


In [ ]:
class DecisionType(Enum):
    """Decision categories used to label simulated logistics contexts."""

    FACILITY_LOCATION = "facility_location"
    ROUTE_OPTIMIZATION = "route_optimization"
    CAPACITY_PLANNING = "capacity_planning"
    SUPPLIER_SELECTION = "supplier_selection"
    DEMAND_ALLOCATION = "demand_allocation"


@dataclass
class HumanExpert:
    """Human decision-maker with experience, confidence, and intuition settings."""

    expert_id: str
    expertise_areas: List[DecisionType]
    experience_level: float
    decision_history: List[Dict[str, Any]] = field(default_factory=list)
    confidence_level: float = 0.8
    intuition_strength: float = 0.7

    def make_decision(
        self, problem_context: Dict[str, Any], decision_type: DecisionType
    ) -> Dict[str, Any]:
        """Generate a synthetic recommendation and record its quality indicator."""
        adjusted_confidence = self.confidence_level
        if decision_type not in self.expertise_areas:
            adjusted_confidence *= 0.6

        experience_bonus = self.experience_level * 0.2
        human_noise = np.random.normal(0, 0.1)
        decision_parameters = {
            "risk_tolerance": float(np.clip(0.3 + human_noise + experience_bonus, 0, 1)),
            "innovation_preference": float(np.clip(0.5 + human_noise, 0, 1)),
            "complexity_handling": float(np.clip(0.6 + experience_bonus, 0, 1)),
        }

        rationality_score = 0.7 + experience_bonus
        intuition_score = self.intuition_strength * 0.3 * np.random.uniform(0.8, 1.2)
        decision_quality = float(
            np.clip(rationality_score * 0.6 + intuition_score * 0.4, 0, 1)
        )
        decision = {
            "expert_id": self.expert_id,
            "decision_type": decision_type.value,
            "decision_parameters": decision_parameters,
            "confidence": adjusted_confidence,
            "decision_quality": decision_quality,
            "reasoning": self._generate_reasoning(decision_type, decision_parameters),
            "timestamp": time.time(),
        }
        self.decision_history.append(decision)
        return decision

    def _generate_reasoning(
        self, decision_type: DecisionType, parameters: Dict[str, float]
    ) -> str:
        """Provide a qualitative explanation for the simulated recommendation."""
        reasoning_templates = {
            DecisionType.FACILITY_LOCATION: "I considered market access, transport infrastructure, and labor availability.",
            DecisionType.ROUTE_OPTIMIZATION: "I considered transport cost, reliability, and delivery-time trade-offs.",
            DecisionType.CAPACITY_PLANNING: "I considered demand variability, risk, and future capacity needs.",
            DecisionType.SUPPLIER_SELECTION: "I considered supplier quality, reliability, and value.",
            DecisionType.DEMAND_ALLOCATION: "I considered demand patterns, service levels, and allocation constraints.",
        }
        reasoning = reasoning_templates[decision_type]
        if parameters["risk_tolerance"] > 0.7:
            reasoning += " The risk-tolerance score favors more innovative options."
        elif parameters["risk_tolerance"] < 0.3:
            reasoning += " The risk-tolerance score favors reliability and stability."
        if parameters["innovation_preference"] > 0.7:
            reasoning += " The innovation-preference score is relatively high."
        return reasoning

    def learn_from_feedback(self, feedback: Dict[str, Any]) -> None:
        """Update human confidence, experience, and intuition settings from feedback."""
        if feedback.get("success", False):
            self.confidence_level = min(1.0, self.confidence_level + 0.02)
            self.experience_level = min(1.0, self.experience_level + 0.01)
        else:
            self.confidence_level = max(0.3, self.confidence_level - 0.01)
            reason = feedback.get("reason", "")
            if "too_conservative" in reason:
                self.intuition_strength = min(1.0, self.intuition_strength + 0.05)
            elif "too_risky" in reason:
                self.intuition_strength = max(0.3, self.intuition_strength - 0.05)


@dataclass
class ExplainableAI:
    """Transparent AI stand-in that generates synthetic recommendation attributes."""

    ai_id: str
    model_type: str = "simulated_rule_based"
    explanation_level: str = "detailed"
    confidence_threshold: float = 0.7
    decision_history: List[Dict[str, Any]] = field(default_factory=list)
    learning_rate: float = 0.1

    def make_decision(
        self, problem_context: Dict[str, Any], decision_type: DecisionType
    ) -> Dict[str, Any]:
        """Simulate data analysis, pattern indicators, and a recommendation."""
        data_analysis = self._analyze_problem_data(problem_context)
        pattern_recognition = self._recognize_patterns(problem_context, decision_type)
        recommendation = self._optimize_solution(problem_context, decision_type)
        model_confidence = self._calculate_model_confidence(decision_type)
        ai_confidence = float(
            np.clip(problem_context.get("data_quality", 0.8) * model_confidence, 0, 1)
        )
        explanation = self._generate_explanation(
            data_analysis, pattern_recognition, recommendation, decision_type
        )
        decision = {
            "ai_id": self.ai_id,
            "decision_type": decision_type.value,
            "decision_parameters": recommendation,
            "confidence": ai_confidence,
            "decision_quality": self._calculate_decision_quality(recommendation),
            "explanation": explanation,
            "data_analysis": data_analysis,
            "timestamp": time.time(),
        }
        self.decision_history.append(decision)
        return decision

    def _analyze_problem_data(self, context: Dict[str, Any]) -> Dict[str, Any]:
        """Collect the context indicators used in the explanation."""
        return {
            "data_completeness": context.get("data_completeness", 0.9),
            "data_accuracy": context.get("data_accuracy", 0.85),
            "data_freshness": context.get("data_freshness", 0.8),
            "complexity_score": context.get("complexity_score", 0.6),
            "uncertainty_level": context.get("uncertainty_level", 0.3),
            "key_factors": ["cost", "time", "quality", "flexibility"],
        }

    def _recognize_patterns(
        self, context: Dict[str, Any], decision_type: DecisionType
    ) -> Dict[str, float]:
        """Generate illustrative pattern indicators rather than learned predictions."""
        return {
            "historical_patterns": float(np.random.uniform(0.6, 0.9)),
            "seasonal_patterns": float(np.random.uniform(0.4, 0.8)),
            "trend_patterns": float(np.random.uniform(0.5, 0.85)),
            "anomaly_patterns": float(np.random.uniform(0.1, 0.3)),
            "pattern_confidence": float(np.random.uniform(0.7, 0.95)),
        }

    def _optimize_solution(
        self, context: Dict[str, Any], decision_type: DecisionType
    ) -> Dict[str, float]:
        """Generate synthetic scores on the shared decision-attribute scale.

        The method name reflects the partnership demonstration; this routine does
        not solve a mathematical optimization problem.
        """
        return {
            "risk_tolerance": float(np.random.uniform(0.2, 0.8)),
            "innovation_preference": float(np.random.uniform(0.3, 0.9)),
            "complexity_handling": float(np.random.uniform(0.5, 0.95)),
        }

    def _calculate_model_confidence(self, decision_type: DecisionType) -> float:
        """Return the illustrative confidence setting for a decision category."""
        type_confidence = {
            DecisionType.FACILITY_LOCATION: 0.85,
            DecisionType.ROUTE_OPTIMIZATION: 0.9,
            DecisionType.CAPACITY_PLANNING: 0.8,
            DecisionType.SUPPLIER_SELECTION: 0.75,
            DecisionType.DEMAND_ALLOCATION: 0.88,
        }
        return type_confidence.get(decision_type, 0.8)

    def _calculate_decision_quality(self, parameters: Dict[str, float]) -> float:
        """Use the mean of the shared attribute scores as a quality proxy."""
        return float(np.mean(list(parameters.values()))) if parameters else 0.5

    def _generate_explanation(
        self,
        data_analysis: Dict[str, Any],
        pattern_recognition: Dict[str, Any],
        optimization_result: Dict[str, float],
        decision_type: DecisionType,
    ) -> str:
        """Explain the synthetic indicators and recommendation scores."""
        attributes = ", ".join(
            f"{name.replace('_', ' ')}={value:.2f}"
            for name, value in optimization_result.items()
        )
        return (
            f"For {decision_type.value.replace('_', ' ')}, the simulation considered "
            f"{len(data_analysis['key_factors'])} factors, with "
            f"{data_analysis['data_completeness']:.1%} data completeness and "
            f"{data_analysis['data_accuracy']:.1%} data accuracy. The synthetic "
            f"pattern-confidence indicator is {pattern_recognition['pattern_confidence']:.1%}; "
            f"the anomaly indicator is {pattern_recognition['anomaly_patterns']:.1%}. "
            f"Shared recommendation attributes: {attributes}. These values are "
            "illustrative scores, not optimized operational decisions."
        )

    def learn_from_feedback(self, feedback: Dict[str, Any]) -> None:
        """Adjust simulation settings; this does not train model parameters."""
        if feedback.get("success", False):
            self.learning_rate = min(0.2, self.learning_rate * 1.1)
        else:
            self.learning_rate = max(0.05, self.learning_rate * 0.9)
            reason = feedback.get("reason", "")
            if "unclear_explanation" in reason:
                levels = ["minimal", "standard", "detailed"]
                if self.explanation_level in levels[:-1]:
                    self.explanation_level = levels[levels.index(self.explanation_level) + 1]
            elif "too_detailed" in reason:
                levels = ["minimal", "standard", "detailed"]
                if self.explanation_level in levels[1:]:
                    self.explanation_level = levels[levels.index(self.explanation_level) - 1]


@dataclass
class TrustFramework:
    """Calculate a bounded trust proxy and summarize relationship indicators."""

    framework_id: str
    trust_history: List[Dict[str, Any]] = field(default_factory=list)
    trust_threshold: float = 0.7
    relationship_metrics: Dict[str, Any] = field(default_factory=dict)

    def calculate_trust_score(
        self,
        human_expert: HumanExpert,
        ai_system: ExplainableAI,
        decision_context: Dict[str, Any],
    ) -> float:
        """Compute the normalized weighted trust score defined above."""
        human_competence = (
            human_expert.decision_history[-1]["decision_quality"]
            if human_expert.decision_history
            else 0.5
        )
        ai_competence = (
            ai_system.decision_history[-1]["decision_quality"]
            if ai_system.decision_history
            else 0.5
        )
        human_integrity_proxy = human_expert.confidence_level
        ai_integrity_proxy = (
            ai_system.decision_history[-1]["confidence"]
            if ai_system.decision_history
            else 0.5
        )
        shared_values = decision_context.get("shared_values", 0.8)
        goal_alignment = decision_context.get("goal_alignment", 0.9)
        benevolence_proxy = (shared_values + goal_alignment) / 2
        weighted_sum = (
            0.3 * human_competence
            + 0.3 * ai_competence
            + 0.2 * human_integrity_proxy
            + 0.2 * ai_integrity_proxy
            + 0.2 * benevolence_proxy
        )
        return float(np.clip(weighted_sum / 1.2, 0, 1))

    def update_relationship_metrics(
        self, trust_score: float, collaboration_outcome: float
    ) -> None:
        """Store current trust and outcome indicators for the partnership report."""
        self.relationship_metrics["trust_score"] = trust_score
        self.relationship_metrics["collaboration_outcome"] = collaboration_outcome
        self.relationship_metrics["relationship_strength"] = (
            trust_score + collaboration_outcome
        ) / 2
        if collaboration_outcome > 0.8 and trust_score > 0.7:
            satisfaction = "high"
        elif collaboration_outcome > 0.6 and trust_score > 0.5:
            satisfaction = "medium"
        else:
            satisfaction = "low"
        self.relationship_metrics["satisfaction"] = satisfaction

    def assess_partnership_quality(self) -> Dict[str, Any]:
        """Map current indicators to a qualitative partnership assessment."""
        if not self.relationship_metrics:
            return {
                "partnership_status": "unknown",
                "trust_level": 0.0,
                "collaboration_effectiveness": 0.0,
                "recommendations": ["Establish initial partnership interactions"],
            }

        trust_level = self.relationship_metrics.get("trust_score", 0.0)
        effectiveness = self.relationship_metrics.get("collaboration_outcome", 0.0)
        if trust_level > 0.8 and effectiveness > 0.8:
            status = "excellent"
            recommendations = [
                "Continue the current partnership approach",
                "Consider expanding the collaboration scope",
            ]
        elif trust_level > 0.6 and effectiveness > 0.6:
            status = "good"
            recommendations = [
                "Maintain the current partnership",
                "Monitor opportunities for improvement",
            ]
        elif trust_level > 0.4 and effectiveness > 0.4:
            status = "developing"
            recommendations = [
                "Increase interaction frequency",
                "Improve communication clarity",
                "Build shared understanding",
            ]
        else:
            status = "needs_improvement"
            recommendations = [
                "Address trust concerns",
                "Review decision-quality indicators",
                "Enhance communication",
                "Realign goals",
            ]
        return {
            "partnership_status": status,
            "trust_level": trust_level,
            "collaboration_effectiveness": effectiveness,
            "relationship_strength": self.relationship_metrics.get(
                "relationship_strength", 0.0
            ),
            "satisfaction": self.relationship_metrics.get("satisfaction", "unknown"),
            "recommendations": recommendations,
        }


@dataclass
class HumanAIPartnership:
    """Coordinate recommendations, trust assessment, fusion, and feedback."""

    partnership_id: str
    human_expert: HumanExpert
    ai_system: ExplainableAI
    trust_framework: TrustFramework
    collaboration_history: List[Dict[str, Any]] = field(default_factory=list)
    partnership_metrics: Dict[str, float] = field(default_factory=dict)

    def collaborate_on_decision(
        self, problem_context: Dict[str, Any], decision_type: DecisionType
    ) -> Dict[str, Any]:
        """Run one complete recommendation, trust, and decision-fusion cycle."""
        human_decision = self.human_expert.make_decision(problem_context, decision_type)
        ai_decision = self.ai_system.make_decision(problem_context, decision_type)
        trust_score = self.trust_framework.calculate_trust_score(
            self.human_expert, self.ai_system, problem_context
        )
        collaborative_decision = self._fuse_decisions(
            human_decision, ai_decision, trust_score
        )
        record = {
            "timestamp": time.time(),
            "decision_type": decision_type.value,
            "human_decision": human_decision,
            "ai_decision": ai_decision,
            "collaborative_decision": collaborative_decision,
            "trust_score": trust_score,
            "collaboration_effectiveness": collaborative_decision["decision_quality"],
        }
        self.collaboration_history.append(record)
        self.trust_framework.update_relationship_metrics(
            trust_score, collaborative_decision["decision_quality"]
        )
        return collaborative_decision

    def _fuse_decisions(
        self,
        human_decision: Dict[str, Any],
        ai_decision: Dict[str, Any],
        trust_score: float,
    ) -> Dict[str, Any]:
        """Fuse matching recommendation attributes with confidence-adjusted weights."""
        if not 0 <= trust_score <= 1:
            raise ValueError("trust_score must be between 0 and 1")

        if trust_score > 0.8:
            base_human_weight, base_ai_weight = 0.3, 0.7
        elif trust_score > 0.5:
            base_human_weight, base_ai_weight = 0.5, 0.5
        else:
            base_human_weight, base_ai_weight = 0.7, 0.3

        human_confidence = float(human_decision.get("confidence", 0.5))
        ai_confidence = float(ai_decision.get("confidence", 0.5))
        human_weight = base_human_weight * human_confidence
        ai_weight = base_ai_weight * ai_confidence
        total_weight = human_weight + ai_weight
        if total_weight == 0:
            human_weight = ai_weight = 0.5
        else:
            human_weight /= total_weight
            ai_weight /= total_weight

        human_params = human_decision.get("decision_parameters", {})
        ai_params = ai_decision.get("decision_parameters", {})
        if human_params.keys() != ai_params.keys():
            raise ValueError("Human and AI recommendations must use matching attributes")
        fused_parameters = {
            key: human_weight * human_params[key] + ai_weight * ai_params[key]
            for key in human_params
        }
        human_quality = human_decision.get("decision_quality", 0.5)
        ai_quality = ai_decision.get("decision_quality", 0.5)
        fused_quality = human_weight * human_quality + ai_weight * ai_quality
        return {
            "partnership_id": self.partnership_id,
            "decision_type": human_decision.get("decision_type"),
            "decision_parameters": fused_parameters,
            "decision_quality": fused_quality,
            "collaboration_effectiveness": fused_quality,
            "human_weight": human_weight,
            "ai_weight": ai_weight,
            "trust_score": trust_score,
            "collaborative_reasoning": self._generate_collaborative_reasoning(
                human_decision, ai_decision, human_weight, ai_weight
            ),
            "timestamp": time.time(),
        }

    def _generate_collaborative_reasoning(
        self,
        human_decision: Dict[str, Any],
        ai_decision: Dict[str, Any],
        human_weight: float,
        ai_weight: float,
    ) -> str:
        """Make each participant's contribution and its weight explicit."""
        human_reasoning = human_decision.get("reasoning", "Human expert analysis")
        ai_explanation = ai_decision.get("explanation", "AI simulation analysis")
        return (
            f"The fused recommendation assigns {human_weight:.1%} weight to the human "
            f"and {ai_weight:.1%} to the AI simulation. Human perspective: "
            f"{human_reasoning} AI perspective: {ai_explanation}"
        )

    def provide_feedback(self, feedback: Dict[str, Any]) -> None:
        """Pass the same simulated outcome feedback to both participants."""
        self.human_expert.learn_from_feedback(feedback)
        self.ai_system.learn_from_feedback(feedback)
        self._update_partnership_metrics()

    def _update_partnership_metrics(self) -> None:
        """Summarize up to the five most recent collaboration records."""
        if not self.collaboration_history:
            return
        recent = self.collaboration_history[-5:]
        average_trust = float(np.mean([item["trust_score"] for item in recent]))
        average_effectiveness = float(
            np.mean([item["collaboration_effectiveness"] for item in recent])
        )
        self.partnership_metrics = {
            "average_trust_score": average_trust,
            "average_effectiveness": average_effectiveness,
            "collaboration_count": len(self.collaboration_history),
            "partnership_strength": (average_trust + average_effectiveness) / 2,
            "learning_rate": (
                self.human_expert.experience_level + self.ai_system.learning_rate
            )
            / 2,
        }

    def generate_partnership_report(self) -> Dict[str, Any]:
        """Return participant details, recent metrics, and a qualitative assessment."""
        assessment = self.trust_framework.assess_partnership_quality()
        history = self.collaboration_history
        report = {
            "report_timestamp": time.time(),
            "partnership_id": self.partnership_id,
            "participants": {
                "human_expert": {
                    "id": self.human_expert.expert_id,
                    "experience_level": self.human_expert.experience_level,
                    "confidence_level": self.human_expert.confidence_level,
                    "expertise_areas": [
                        area.value for area in self.human_expert.expertise_areas
                    ],
                    "decisions_made": len(self.human_expert.decision_history),
                },
                "ai_system": {
                    "id": self.ai_system.ai_id,
                    "model_type": self.ai_system.model_type,
                    "explanation_level": self.ai_system.explanation_level,
                    "learning_rate": self.ai_system.learning_rate,
                    "decisions_made": len(self.ai_system.decision_history),
                },
            },
            "partnership_assessment": assessment,
            "collaboration_metrics": self.partnership_metrics,
            "collaboration_history_summary": {
                "total_collaborations": len(history),
                "decision_types": sorted({item["decision_type"] for item in history}),
                "average_trust_score": float(
                    np.mean([item["trust_score"] for item in history])
                )
                if history
                else 0.0,
                "average_effectiveness": float(
                    np.mean([item["collaboration_effectiveness"] for item in history])
                )
                if history
                else 0.0,
            },
            "recommendations": [],
        }
        if assessment["trust_level"] < 0.5:
            report["recommendations"].append(
                "Review trust indicators and strengthen transparent communication"
            )
        if assessment["collaboration_effectiveness"] < 0.6:
            report["recommendations"].append(
                "Review the collaboration process and decision-fusion assumptions"
            )
        if self.human_expert.experience_level < 0.7:
            report["recommendations"].append(
                "Provide further development opportunities for the human expert"
            )
        if self.ai_system.learning_rate < 0.1:
            report["recommendations"].append(
                "Review the simulated feedback and learning-rate settings"
            )
        if len(history) < 10:
            report["recommendations"].append(
                "Collect more simulated interactions before drawing broad conclusions"
            )
        return report

In [ ]:
print("Initializing the human–AI partnership simulation...")
print("=" * 52)

human_expert = HumanExpert(
    expert_id="logistics_expert_001",
    expertise_areas=[
        DecisionType.FACILITY_LOCATION,
        DecisionType.ROUTE_OPTIMIZATION,
        DecisionType.CAPACITY_PLANNING,
    ],
    experience_level=0.75,
    confidence_level=0.8,
    intuition_strength=0.7,
)

ai_system = ExplainableAI(
    ai_id="network_ai_001",
    model_type="simulated_rule_based",
    explanation_level="detailed",
    confidence_threshold=0.7,
    learning_rate=0.1,
)

trust_framework = TrustFramework(
    framework_id="trust_framework_001",
    trust_threshold=0.7,
)

partnership = HumanAIPartnership(
    partnership_id="human_ai_partnership_001",
    human_expert=human_expert,
    ai_system=ai_system,
    trust_framework=trust_framework,
)

print(f"Human expert: {human_expert.expert_id}")
print(f"  Expertise areas: {[area.value for area in human_expert.expertise_areas]}")
print(f"  Experience level: {human_expert.experience_level:.2f}")
print(f"  Confidence level: {human_expert.confidence_level:.2f}")
print(f"  Intuition strength: {human_expert.intuition_strength:.2f}")
print(f"\nAI simulation: {ai_system.ai_id}")
print(f"  Model description: {ai_system.model_type}")
print(f"  Explanation level: {ai_system.explanation_level}")
print(f"  Learning-rate proxy: {ai_system.learning_rate:.3f}")
print(f"  Confidence threshold setting: {ai_system.confidence_threshold:.2f}")
print(f"\nTrust framework: {trust_framework.framework_id}")
print(f"  Trust threshold setting: {trust_framework.trust_threshold:.2f}")
print(f"\nPartnership: {partnership.partnership_id}")
print(f"  Recorded collaborations: {len(partnership.collaboration_history)}")
print("\nThe simulation components are ready.")

In [ ]:
def run_partnership_simulation(
    partnership: HumanAIPartnership, num_decisions: int = 20
) -> Dict[str, Any]:
    """Run synthetic decisions, feedback updates, and performance summaries."""
    if num_decisions <= 0:
        raise ValueError("num_decisions must be a positive integer")

    print("Running the human–AI partnership simulation...")
    print(f"Number of decisions: {num_decisions}")
    print("=" * 42)
    simulation_results = {
        "decisions": [],
        "trust_evolution": [],
        "performance_metrics": {},
        "learning_progress": [],
    }
    decision_types = list(DecisionType)

    for i in range(num_decisions):
        decision_type = decision_types[i % len(decision_types)]
        problem_context = {
            "decision_id": f"decision_{i + 1}",
            "data_quality": float(np.random.uniform(0.7, 0.95)),
            "data_completeness": float(np.random.uniform(0.8, 1.0)),
            "data_accuracy": float(np.random.uniform(0.85, 0.98)),
            "complexity_score": float(np.random.uniform(0.3, 0.8)),
            "uncertainty_level": float(np.random.uniform(0.1, 0.4)),
            "shared_values": float(np.random.uniform(0.7, 0.95)),
            "goal_alignment": float(np.random.uniform(0.8, 0.98)),
        }
        decision = partnership.collaborate_on_decision(problem_context, decision_type)

        # Outcomes are synthetic labels, not observations of real-world performance.
        actual_outcome = float(np.random.uniform(0.5, 1.0))
        success = actual_outcome > 0.6
        feedback = {
            "success": success,
            "actual_outcome": actual_outcome,
            "decision_quality": decision["decision_quality"],
            "improvement_needed": not success,
            "reason": "performance_evaluation",
        }
        if not success:
            if decision["decision_quality"] < 0.6:
                feedback["reason"] = "low_decision_quality"
            elif decision["trust_score"] < 0.5:
                feedback["reason"] = "low_trust_score"
            else:
                feedback["reason"] = "random_factors"
        partnership.provide_feedback(feedback)

        simulation_results["decisions"].append(
            {
                "decision_number": i + 1,
                "decision_type": decision_type.value,
                "decision_quality": decision["decision_quality"],
                "trust_score": decision["trust_score"],
                "human_weight": decision["human_weight"],
                "ai_weight": decision["ai_weight"],
                "collaboration_effectiveness": decision["collaboration_effectiveness"],
                "actual_outcome": actual_outcome,
                "success": success,
            }
        )
        simulation_results["trust_evolution"].append(
            {
                "decision_number": i + 1,
                "trust_score": decision["trust_score"],
                "human_confidence": partnership.human_expert.confidence_level,
                "ai_confidence": partnership.ai_system.decision_history[-1]["confidence"],
            }
        )
        simulation_results["learning_progress"].append(
            {
                "decision_number": i + 1,
                "human_experience": partnership.human_expert.experience_level,
                "ai_learning_rate": partnership.ai_system.learning_rate,
                "partnership_strength": partnership.partnership_metrics.get(
                    "partnership_strength", 0.5
                ),
            }
        )
        if (i + 1) % 5 == 0:
            print(f"Decision {i + 1}: {decision_type.value.replace('_', ' ').title()}")
            print(f"  Decision-quality indicator: {decision['decision_quality']:.3f}")
            print(f"  Trust score: {decision['trust_score']:.3f}")
            print(
                f"  Human weight: {decision['human_weight']:.2f}; "
                f"AI weight: {decision['ai_weight']:.2f}"
            )
            print(f"  Synthetic outcome label: {'Success' if success else 'Not success'}")

    decisions = simulation_results["decisions"]
    successful = sum(item["success"] for item in decisions)
    simulation_results["performance_metrics"] = {
        "total_decisions": len(decisions),
        "successful_decisions": successful,
        "success_rate": successful / len(decisions),
        "average_decision_quality": float(
            np.mean([item["decision_quality"] for item in decisions])
        ),
        "average_trust_score": float(
            np.mean([item["trust_score"] for item in decisions])
        ),
        "average_human_weight": float(
            np.mean([item["human_weight"] for item in decisions])
        ),
        "average_ai_weight": float(
            np.mean([item["ai_weight"] for item in decisions])
        ),
        "average_effectiveness": float(
            np.mean([item["collaboration_effectiveness"] for item in decisions])
        ),
    }
    return simulation_results


simulation_results = run_partnership_simulation(partnership, num_decisions=25)
metrics = simulation_results["performance_metrics"]
print("\nSimulation summary (synthetic indicators):")
print(f"Total decisions: {metrics['total_decisions']}")
print(
    f"Synthetic successes: {metrics['successful_decisions']} "
    f"({metrics['success_rate']:.1%})"
)
print(f"Average decision-quality indicator: {metrics['average_decision_quality']:.3f}")
print(f"Average trust score: {metrics['average_trust_score']:.3f}")
print(f"Average human weight: {metrics['average_human_weight']:.2f}")
print(f"Average AI weight: {metrics['average_ai_weight']:.2f}")
print(f"Average collaboration-effectiveness indicator: {metrics['average_effectiveness']:.3f}")

final_report = partnership.generate_partnership_report()
assessment = final_report["partnership_assessment"]
print("\nPartnership assessment:")
print(f"Status: {assessment['partnership_status'].replace('_', ' ').title()}")
print(f"Trust indicator: {assessment['trust_level']:.3f}")
print(f"Effectiveness indicator: {assessment['collaboration_effectiveness']:.3f}")
print(f"Relationship-strength indicator: {assessment['relationship_strength']:.3f}")
print(f"Satisfaction category: {assessment['satisfaction'].title()}")
print("Recommendations:")
for number, recommendation in enumerate(assessment["recommendations"], start=1):
    print(f"  {number}. {recommendation}")

participants = final_report["participants"]
human = participants["human_expert"]
ai = participants["ai_system"]
print("\nHuman expert details:")
print(f"  ID: {human['id']}")
print(f"  Experience level: {human['experience_level']:.2f}")
print(f"  Confidence level: {human['confidence_level']:.2f}")
print(f"  Expertise areas: {', '.join(human['expertise_areas'])}")
print(f"  Decisions made: {human['decisions_made']}")
print("AI simulation details:")
print(f"  ID: {ai['id']}")
print(f"  Model description: {ai['model_type']}")
print(f"  Explanation level: {ai['explanation_level']}")
print(f"  Learning-rate proxy: {ai['learning_rate']:.3f}")
print(f"  Decisions made: {ai['decisions_made']}")

summary = final_report["collaboration_history_summary"]
print("\nCollaboration history summary:")
print(f"  Total collaborations: {summary['total_collaborations']}")
print(f"  Decision categories: {', '.join(summary['decision_types'])}")
print(f"  Mean trust indicator: {summary['average_trust_score']:.3f}")
print(f"  Mean effectiveness indicator: {summary['average_effectiveness']:.3f}")

In [ ]:
def visualize_partnership_performance(
    partnership: HumanAIPartnership, simulation_results: Dict[str, Any]
) -> Dict[str, Any]:
    """Plot simulated trends and print descriptive summaries of the indicators."""
    decisions = simulation_results["decisions"]
    trust_evolution = simulation_results["trust_evolution"]
    learning_progress = simulation_results["learning_progress"]
    if not decisions:
        raise ValueError("simulation_results must contain at least one decision")

    decision_numbers = [item["decision_number"] for item in decisions]
    trust_scores = [item["trust_score"] for item in trust_evolution]
    human_confidence = [item["human_confidence"] for item in trust_evolution]
    ai_confidence = [item["ai_confidence"] for item in trust_evolution]
    qualities = [item["decision_quality"] for item in decisions]
    successes = [int(item["success"]) for item in decisions]
    human_weights = [item["human_weight"] for item in decisions]
    ai_weights = [item["ai_weight"] for item in decisions]
    human_experience = [item["human_experience"] for item in learning_progress]
    ai_learning = [item["ai_learning_rate"] for item in learning_progress]
    partnership_strength = [item["partnership_strength"] for item in learning_progress]

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    ax1, ax2, ax3, ax4 = axes.flat

    ax1.set_title("Trust and Confidence Indicators")
    ax1.plot(decision_numbers, trust_scores, "b-o", linewidth=2, label="Trust score")
    ax1.plot(
        decision_numbers,
        human_confidence,
        "g--",
        linewidth=2,
        label="Human confidence",
    )
    ax1.plot(
        decision_numbers,
        ai_confidence,
        "r--",
        linewidth=2,
        label="AI confidence",
    )
    ax1.axhline(
        y=partnership.trust_framework.trust_threshold,
        color="orange",
        linestyle=":",
        label="Configured trust threshold",
    )
    ax1.set_xlabel("Decision number")
    ax1.set_ylabel("Score (0–1)")
    ax1.set_ylim(0, 1)
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.set_title("Decision-Quality Indicator and Synthetic Outcome")
    ax2_twin = ax2.twinx()
    quality_line = ax2.plot(
        decision_numbers, qualities, "b-o", linewidth=2, label="Quality indicator"
    )[0]
    success_line = ax2_twin.plot(
        decision_numbers, successes, "r-s", linewidth=2, label="Synthetic success label"
    )[0]
    ax2.set_xlabel("Decision number")
    ax2.set_ylabel("Quality indicator (0–1)", color="b")
    ax2.tick_params(axis="y", labelcolor="b")
    ax2.set_ylim(0, 1)
    ax2_twin.set_ylabel("Synthetic label (1 = success, 0 = not success)", color="r")
    ax2_twin.tick_params(axis="y", labelcolor="r")
    ax2_twin.set_ylim(-0.1, 1.1)
    ax2.legend([quality_line, success_line], [
        quality_line.get_label(), success_line.get_label()
    ], loc="upper left")
    ax2.grid(True, alpha=0.3)

    ax3.set_title("Confidence-Adjusted Decision Weights")
    ax3.plot(decision_numbers, human_weights, "g-o", linewidth=2, label="Human weight")
    ax3.plot(decision_numbers, ai_weights, color="orange", marker="s", label="AI weight")
    ax3.axhline(y=0.5, color="red", linestyle="--", label="Equal weight")
    ax3.set_xlabel("Decision number")
    ax3.set_ylabel("Weight (0–1)")
    ax3.set_ylim(0, 1)
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    ax4.set_title("Learning and Partnership Indicators")
    ax4.plot(
        decision_numbers, human_experience, "g-o", linewidth=2, label="Human experience"
    )
    ax4.plot(
        decision_numbers, ai_learning, color="orange", marker="s", label="AI learning-rate proxy"
    )
    ax4.plot(
        decision_numbers,
        partnership_strength,
        "b-^",
        linewidth=2,
        label="Partnership-strength indicator",
    )
    ax4.set_xlabel("Decision number")
    ax4.set_ylabel("Score or rate (0–1)")
    ax4.set_ylim(0, 1)
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    fig.suptitle("Human–AI Partnership Simulation", fontsize=16)
    fig.tight_layout()
    plt.show()

    initial_trust = trust_scores[0]
    final_trust = trust_scores[-1]
    trust_change = final_trust - initial_trust
    average_quality = float(np.mean(qualities))
    window_size = min(5, len(qualities))
    first_window_average = float(np.mean(qualities[:window_size]))
    recent_average = float(np.mean(qualities[-window_size:]))
    success_rate = float(np.mean(successes))
    final_human_weight = human_weights[-1]
    final_ai_weight = ai_weights[-1]
    final_partnership = partnership_strength[-1]

    print("\nHuman–AI partnership simulation analysis:")
    print("=" * 44)
    print("Trust indicator:")
    print(f"  Initial score: {initial_trust:.3f}")
    print(f"  Final score: {final_trust:.3f}")
    print(f"  Change in score: {trust_change:+.3f}")
    if final_trust > 0.8:
        print("  Category: High under the configured descriptive thresholds")
    elif final_trust > 0.6:
        print("  Category: Moderate under the configured descriptive thresholds")
    elif final_trust > 0.4:
        print("  Category: Developing under the configured descriptive thresholds")
    else:
        print("  Category: Low under the configured descriptive thresholds")

    print("\nDecision-quality indicator:")
    print(f"  Mean across decisions: {average_quality:.3f}")
    print(f"  First {window_size}-decision mean: {first_window_average:.3f}")
    print(f"  Last {window_size}-decision mean: {recent_average:.3f}")
    if np.isclose(recent_average, first_window_average):
        quality_trend = "similar across the two windows"
    elif recent_average > first_window_average:
        quality_trend = "higher in the last window"
    else:
        quality_trend = "lower in the last window"
    print(f"  Descriptive comparison: {quality_trend}")

    print("\nFinal fusion weights:")
    print(f"  Human: {final_human_weight:.3f}")
    print(f"  AI: {final_ai_weight:.3f}")
    print(f"  Weight sum: {final_human_weight + final_ai_weight:.3f}")
    if abs(final_human_weight - final_ai_weight) < 0.1:
        print("  Balance: approximately equal")
    elif final_human_weight > final_ai_weight:
        print("  Balance: human-weighted")
    else:
        print("  Balance: AI-weighted")

    print("\nFeedback-loop indicators:")
    print(f"  Final human experience: {human_experience[-1]:.3f}")
    print(f"  Final AI learning-rate proxy: {ai_learning[-1]:.3f}")
    print(f"  Final partnership-strength indicator: {final_partnership:.3f}")
    print(f"  Synthetic outcome success fraction: {success_rate:.1%}")
    print("  These are simulated indicators, not evidence of real-world performance.")

    return {
        "initial_trust": initial_trust,
        "final_trust": final_trust,
        "trust_change": trust_change,
        "average_quality": average_quality,
        "first_window_quality": first_window_average,
        "recent_window_quality": recent_average,
        "final_human_weight": final_human_weight,
        "final_ai_weight": final_ai_weight,
        "final_partnership_strength": final_partnership,
        "synthetic_success_rate": success_rate,
    }


visualization_summary = visualize_partnership_performance(
    partnership, simulation_results
)

## Role of the Human–AI Partnership

A human–AI partnership is useful when a decision process needs both domain judgment and computational support. In this notebook, trust-based weighting, transparent explanations, and feedback are demonstrated as components of a **simulation framework**. The synthetic example does not establish that a partnership is superior to a human-only or AI-only process; that conclusion would require comparable baselines, a defined evaluation design, and suitable empirical data.

### Potential benefits and limitations

**Potential benefits represented by the framework**
- Human and AI contributions are made explicit through separate recommendations and weights.
- Explanations expose the synthetic indicators used by the AI stand-in.
- Feedback updates selected human and AI simulation settings.
- Trust and collaboration indicators make assumptions visible for discussion.

**Limitations to keep in mind**
- The AI component is not a trained model and does not optimize a network design.
- Synthetic decision attributes and outcomes are not calibrated against operational data.
- The trust score compresses complex social and organizational factors into a weighted indicator.
- The feedback updates do not demonstrate learning gains or causal improvement.
- The notebook does not model implementation cost, communication delay, accountability, or operational feasibility.

### When this framework may be relevant
The partnership structure can serve as a starting point for examining decision support in complex or high-stakes logistics settings, particularly when human oversight and explanations matter. Applying it to an operational network-design decision requires a well-defined decision model, appropriate data, feasibility checks, and evaluation against relevant baselines; those elements are outside this simulation.

### Reading the simulation
1. Interpret trust, quality, and effectiveness as model-defined indicators, not objective measures.
2. Check how trust and confidence affect the human and AI weights in each decision.
3. Compare the first and last decision windows descriptively; a difference alone does not demonstrate learning or causality.
4. Treat the success fraction as a property of the synthetic outcome generator and its threshold, not as a forecast of real performance.

### Partnership components shown
- **Explainable AI stand-in:** provides a readable account of synthetic context and recommendation scores.
- **Trust framework:** combines competence, confidence-based proxies, and shared-goal alignment.
- **Decision fusion:** forms weighted averages of matching recommendation attributes.
- **Feedback loop:** updates selected simulation settings after synthetic outcomes.
- **Performance summaries:** report the indicators and their trends without making unsupported comparative claims.